# M2/M3 Episodic + Procedural Memory

本轮分别只开启 episodic 和 procedural memory，避免把路由改善混在一起：

- episodic：从历史邮件处理示例学习路由；
- procedural：从反馈规则更新后续指令。

In [ ]:
import json

from email_memory_lab import (
    AgentConfig,
    EmailAgent,
    LocalStore,
    evaluate_config,
    find_project_root,
    load_labeled_emails,
    seed_memories,
    start_run,
)

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"
cases = load_labeled_emails(DATA / "emails.json")
store = LocalStore()
seed_memories(store, DATA / "memories.json", "user-a")
run = start_run(PROJECT_ROOT, "03-episodic-procedural", {"memory": "episodic+procedural"})

In [ ]:
configs = {
    "baseline": AgentConfig(),
    "episodic": AgentConfig(use_episodic=True),
    "procedural": AgentConfig(use_procedural=True),
}

results = {label: evaluate_config(cases, store, "user-a", config) for label, config in configs.items()}
for label, result in results.items():
    print(label, json.dumps(result["summary"], ensure_ascii=False))
    print("failures:", [row["email_id"] for row in result["failures"]])

## 查看单条记忆如何改变决策

不要只看总分，要检查命中的记忆和最终动作。

In [ ]:
case = next(case for case in cases if case.email.id == "vendor-pricing")
result = EmailAgent(store, "user-a", AgentConfig(use_episodic=True)).handle(case.email)
print(json.dumps({
    "action": result.action,
    "used_memory": result.used_memory,
    "context": result.context,
}, ensure_ascii=False, indent=2))

In [ ]:
case = next(case for case in cases if case.email.id == "event-invite")
result = EmailAgent(store, "user-a", AgentConfig(use_procedural=True)).handle(case.email)
print(json.dumps({
    "action": result.action,
    "used_memory": result.used_memory,
    "context": result.context,
}, ensure_ascii=False, indent=2))

## 检查点

- episodic 是否只修正了与示例相似的 vendor 邮件？
- procedural 规则是否误伤其他发件人？
- 两类记忆都开启后，是否出现相互覆盖？
- 为什么 procedural 规则的来源、版本和回滚点必须被记录？

In [ ]:
path = run.save("episodic-procedural-comparison.json", results)
print(f"Saved: {path}")